Car Acceleration prediction Mode
Author: Andreas Loizias
Date: 2026/01/26
This model uses the dataset by Jahaidul Islam of a large number of entries with car specifications to predict a car's 0-100 Km/h acceleration time.
It has been created with the help of ChatGPT but not created entirely by it.

In [ ]:
# =========================
# Core libraries
# =========================
import numpy as np
import pandas as pd

# =========================
# Visualization
# =========================
import matplotlib.pyplot as plt
import seaborn as sns

# =========================
# Machine Learning
# =========================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# =========================
# Settings & reproducibility
# =========================
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

np.random.seed(22)
tf.random.set_seed(22)

sns.set(style="whitegrid")


In [ ]:
# Load dataset (update path/name for full dataset)
df = pd.read_csv("/kaggle/input/car-dataset-reduced/car_dataset_reduced.csv")

# Quick look
print("Original shape:", df.shape)
df.head()

In [ ]:
#Drop rows that have missing values for the listed columns
df2 = df.replace(r"^\s*$", np.nan, regex=True)  # turn "" or "   " into NaN
target_col = df2.columns[-1]  # last column = acceleration

critical_cols = [
    target_col,
    "curb_weight_kg",
    "maximum_torque_n_m",
    "injection_type",
    "cylinder_layout",
    "number_of_cylinders",
    "engine_type",
    "valves_per_cylinder",
    "boost_type",
    "capacity_cm3",
    "engine_hp",
    "drive_wheels",
    "number_of_gears",
    "transmission",
    "acceleration_0_100_km/h_s"
]


df_clean = df2.dropna(subset=critical_cols).reset_index(drop=True)

df_clean.shape[0]
df_clean.info()  # also shows non-null counts per column
df_clean.head()

In [ ]:
#make sure the columns that should be numeric really are
numeric_check = [
    "Year_from",
    "curb_weight_kg",
    "maximum_torque_n_m",
    "number_of_cylinders",
    "valves_per_cylinder",
    "capacity_cm3",
    "engine_hp",
    "number_of_gears",
    "acceleration_0_100_km/h_s"
]

df_clean[numeric_check].dtypes

In [ ]:
#some columns are object type and should be float so convert them
cols_to_float = [
    "curb_weight_kg",
    "maximum_torque_n_m",
    "capacity_cm3"
]

for col in cols_to_float:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

df_clean[cols_to_float].dtypes


In [ ]:
#Adding calculated columns for hp/weight and torque/weight which might be quite important contributors to acceleration time
df_clean["hp_per_kg"] = df_clean["engine_hp"] / df_clean["curb_weight_kg"]
df_clean["torque_per_kg"] = df_clean["maximum_torque_n_m"] / df_clean["curb_weight_kg"]

#move the target column back to the end
target_col = "acceleration_0_100_km/h_s"
df_clean = df_clean[[c for c in df_clean.columns if c != target_col] + [target_col]]
df_clean.head()


We will now generate some graphs to get a feel for the data

In [ ]:
# Get a total count for the target column
plt.figure(figsize=(8, 5))
plt.hist(df_clean["acceleration_0_100_km/h_s"], bins=40)
plt.xlabel("0–100 km/h (seconds)")
plt.ylabel("Count")
plt.title("Distribution of 0–100 km/h Acceleration")
plt.show()
# below we can see that it trails off to the right but over is bell shaped which makes sense

In [ ]:
# Generate graph for the power to weight column which intuitevely is the most important one
plt.figure(figsize=(8, 5))
plt.scatter(
    df_clean["hp_per_kg"],
    df_clean["acceleration_0_100_km/h_s"],
    alpha=0.4
)
plt.xlabel("Horsepower per kg")
plt.ylabel("0–100 km/h (seconds)")
plt.title("Acceleration vs Power-to-Weight")
plt.show()
#indeed we see that as the horsepower per kg increases the 0-100 time decreases significantly and tails off so it means that 
#the more power per kg slowly reduces the 0-100 time less as it trails off

In [ ]:
# we plot the power and weight vs acceleration seperatly to get a feel which of the two affects acceleration the most
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].scatter(df_clean["engine_hp"], df_clean["acceleration_0_100_km/h_s"], alpha=0.4)
ax[0].set_xlabel("Engine HP")
ax[0].set_ylabel("0–100 km/h (s)")
ax[0].set_title("Acceleration vs Engine HP")

ax[1].scatter(df_clean["curb_weight_kg"], df_clean["acceleration_0_100_km/h_s"], alpha=0.4)
ax[1].set_xlabel("Curb Weight (kg)")
ax[1].set_ylabel("0–100 km/h (s)")
ax[1].set_title("Acceleration vs Weight")

plt.show()
# we can see that the engine HP has a more defined relationship while the weight does not affect it as much

In [ ]:
#plot acceleration vs wheel driven
plt.figure(figsize=(8, 5))
df_clean.boxplot(
    column="acceleration_0_100_km/h_s",
    by="drive_wheels",
    grid=False
)
plt.title("Acceleration by Drive Wheels")
plt.suptitle("")
plt.xlabel("Drive Wheels")
plt.ylabel("0–100 km/h (s)")
plt.show()

In [ ]:
#Check the transmission type
plt.figure(figsize=(8, 5))
df_clean.boxplot(
    column="acceleration_0_100_km/h_s",
    by="transmission",
    grid=False
)
plt.title("Acceleration by Transmission")
plt.suptitle("")
plt.xlabel("Transmission")
plt.ylabel("0–100 km/h (s)")
plt.show()


In [ ]:
#generate a heat map for all columns
num_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns

corr = df_clean[num_cols].corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr)
plt.colorbar()
plt.xticks(range(len(num_cols)), num_cols, rotation=90)
plt.yticks(range(len(num_cols)), num_cols)
plt.title("Numeric Feature Correlation Matrix")
plt.show()
# we can see that engine power, weight, torque massively affect the acceleration
# capacity and valves per cylinder as well but not as much
# the least affeecting ones are yaer and number of gears but we should keep it as it gears increase as years gets bigger (newer cars have more gears) and might help refine the predictions

In [ ]:
#separate target column and attribute columns
x = df_clean.drop("acceleration_0_100_km/h_s", axis=1)
y = df_clean["acceleration_0_100_km/h_s"]

In [ ]:
#one-hot encode catecotrical columns, original categorical columns are dropped automatically
categorical_cols = x.select_dtypes(include=["object", "category"]).columns
X_encoded = pd.get_dummies(x, columns=categorical_cols, drop_first=True)
X_encoded.head()

In [ ]:
#scale numeric columns, exclude ones that are binary
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# select numeric columns
numeric_cols = X_encoded.select_dtypes(include=["int64", "float64"]).columns

# keep only columns that are NOT binary (not just 0 and 1)
cols_to_scale = [
    col for col in numeric_cols
    if not set(X_encoded[col].dropna().unique()).issubset({0, 1})
]

# apply scaling
X_encoded[cols_to_scale] = scaler.fit_transform(X_encoded[cols_to_scale])
X_encoded[cols_to_scale].describe().loc[["mean", "std"]]


In [ ]:
#split data to train data set (80%) and test data set (20%) and shuffle it since the data is sorted in a time manner
#we did not create a separate validation data set so we can use k-fold cross-validation on the training set to tune hyperparameters
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=22,
    shuffle=True
)

print("Training size:", X_train.shape[0])
print("Test size:", X_test.shape[0])


In [ ]:
#Import linear regression model library and define the model
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),   # good default for numeric features
    ("scaler", StandardScaler(with_mean=False)),     # safe if X is sparse due to one-hot
    ("model", Ridge())
])

param_grid = {
    "model__alpha": [0.01, 0.1, 1, 10, 100]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    error_score="raise"   # makes debugging easier if anything else breaks
)

grid.fit(X_train, y_train)

print("Best alpha:", grid.best_params_)
print("Best CV RMSE:", -grid.best_score_)

In [ ]:
#evaluate model using the test set
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Test MAE:", mae)
print("Test RMSE:", rmse)
print("Test R²:", r2)

In [ ]:
#We got a 77% success which is a bit low and the MAE is about 1.11 seconds which means our model's prediction is about 1.11 seconds off the actual acceleration time
# we're trying elastic nets
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

pipeline_en = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
    ("model", ElasticNet(max_iter=5000))
])

param_grid_en = {
    "model__alpha": [0.01, 0.1, 1, 10],
    "model__l1_ratio": [0.2, 0.5, 0.8]
}

grid_en = GridSearchCV(
    pipeline_en,
    param_grid_en,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=1
)

grid_en.fit(X_train, y_train)

print("Best params:", grid_en.best_params_)
print("Best CV RMSE:", -grid_en.best_score_)

In [ ]:
best_en = grid_en.best_estimator_

y_pred_en = best_en.predict(X_test)

mae_en = mean_absolute_error(y_test, y_pred_en)
rmse_en = np.sqrt(mean_squared_error(y_test, y_pred_en))
r2_en = r2_score(y_test, y_pred_en)

print("ElasticNet Test MAE:", mae_en)
print("ElasticNet Test RMSE:", rmse_en)
print("ElasticNet Test R²:", r2_en)

In [ ]:
#The elastic net model did not significantly improve the predictions so we are trying a one fast non-linear model (Hist Gradient Boosting Regressor)
from sklearn.ensemble import HistGradientBoostingRegressor

hgb = HistGradientBoostingRegressor(random_state=42)
hgb.fit(X_train, y_train)

y_pred_hgb = hgb.predict(X_test)

print("R²:", r2_score(y_test, y_pred_hgb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_hgb)))

In [ ]:
y_train_pred = hgb.predict(X_train)

print("Train R²:", r2_score(y_train, y_train_pred))
print("Test R²:", r2_score(y_test, y_pred_hgb))

In [ ]:
#Trying a simple neural network mode
from sklearn.neural_network import MLPRegressor

pipeline_nn = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        max_iter=1000,
        random_state=42,
        early_stopping=True
    ))
])

pipeline_nn.fit(X_train, y_train)

y_pred_nn = pipeline_nn.predict(X_test)

print("NN R²:", r2_score(y_test, y_pred_nn))
print("NN RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_nn)))

The non linear model's prediction are much better than linear regression which is logical considering that the biggest predictor as seen from the plots we did earlier is the power to weight ratio with the acceleration decreasing exponentially as the power to weight ratio increases. The non linear models predicts 94% accurately and the on average the acceleration time predictions are less than a second (~93 second) off.
We also tried a simple neural network which is a bit of an overkill for simple tabular data and it did not perform any better.
The neural network explains about 87% of the differences in acceleration times and predicts within roughly 1.3 seconds on average.